## Импорт библиотек

Рекомендуется делать это задание на [google colab](https://colab.research.google.com/).
Распакуйте архив _additional_files.zip_ и положите в корень к этому ноутбуку все файлы.


In [ ]:
# Рекомендуем положить датасет на гугл диск и подключить его следующей командой:
from google.colab import drive
drive.mount('/content/drive')

# Такой командой можно распаковать архив с данными
!unzip /content/drive/MyDrive/sr/train.zip -d /content/SR-task/

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import torch
import os

## Интерполяция

### Интерполяция одномерного сигнала

**Интерполяция** — это процесс оценки промежуточных значений непрерывного сигнала на основе дискретных сэмплов. Интерполяция широко используется в цифровой обработке сигналов для увеличения или уменьшения их разрешения.

Интерполяционная функция представляет собой специальный вид аппроксимирующей функции. Одним из фундаментальных свойств интерполяционной функции является то, что она должна совпадать с выборочными данными в интерполяционных узлах или выборочных точках. Другими словами, если $ f(x) $ — это обрабатываемая функция, а $ g(x) $ — соответствующая интерполяционная функция, то $ g(x_k) = f(x_k) $ в узлах интерполяции $ x_k $.

Формула для интерполированной функции выглядит следующим образом:

$$
g(x) = \sum_{k} c_k u\left(\frac{x - x_k}{h}\right) \tag{1}
$$

Где:
- $ h $ — шаг семплирования;
- $ x_k $ — узлы интерполяции;
- $ u $ — ядро интерполяции, интерполяционное ядро должно быть симметричным;
- $ g $ — интерполяционная функция;
- $ c_k $ — параметры, зависящие от выборочных данных, которые выбираются так, чтобы выполнялось условие интерполяции $ g(x_k) = f(x_k) $.

Интерполяционное ядро в уравнении _(1)_ переводит дискретные данные в непрерывные функции с помощью операции, аналогичной свёртке. Интерполяционные ядра оказывают значительное влияние на численное поведение интерполяционных функций. Ввиду их влияния на точность и эффективность, интерполяционные ядра могут быть эффективно использованы для создания новых алгоритмов интерполяции.

In [ ]:
from scipy.interpolate import interp1d
EPS = 1e-9

In [ ]:
def interpolate(signal, scale_factor, kernel):
    """
    Интерполирует одномерный сигнал до новой длины с использованием заданного ядра
    без использования явных циклов, используя векторизованные операции NumPy.

    :param signal: Исходный сигнал (1D numpy массив).
    :param scale_factor: Коэффициент масштабирования
    :param kernel: Функция ядра, принимающая массив расстояний и возвращающая веса.
    :return: Интерполированный сигнал (1D numpy массив).
    """
    original_length = len(signal)
    new_length = int(original_length * scale_factor)
    # Позиции новых точек
    new_indices = np.linspace(0, original_length - 1, new_length)
    # Позиции исходных точек
    original_x = np.arange(original_length)
    # Вычисляем расстояния от x до всех узлов x_k: shape (new_length, original_length)
    distances = np.abs(new_indices[:, np.newaxis] - original_x[np.newaxis, :])
    # Применяем ядро: shape (new_length, original_length)
    weights = kernel(distances)
    # Нормализуем веса по строкам (новым точкам)
    weights_sum = np.sum(weights, axis=1, keepdims=True)
    weights_sum[weights_sum == 0] = 1
    weights_normalized = weights / weights_sum
    # Вычисляем интерполированный сигнал: shape (new_length,)
    interpolated = np.dot(weights_normalized, signal)
    return interpolated

In [ ]:
x = np.linspace(-2, 2, num=5)
y = np.abs(x)
SCALE = 2
x_new = np.linspace(-2, 2, num=SCALE*x.shape[0])

#### Метод ближайшего соседа


Ядро интерполяции методом **ближайшего соседа** является одним из самых простых методов интерполяции. Оно основано на том, что значение в новой точке просто принимается равным значению ближайшего узла (точки выборки). В математическом выражении интерполяционное ядро этого метода можно записать как:

$$
u(s) =
\begin{cases}
1, & |s| < 0.5 \\
0, & |s| \geq 0.5
\end{cases}
$$

In [ ]:
def nearest_neighbor_kernel(distance):
    """
    Ядро ближайшего соседа
    Присваивает вес 1 ближайшей точке и 0 остальным

    :param distance: Массив расстояний до исходных точек (shape: (new_length, original_length))
    :return: Массив весов (shape: (new_length, original_length))
    """
    return np.where(distance<0.5,1.0,0.0) # ваш код здесь

In [ ]:
assert np.linalg.norm(interpolate(y, SCALE, nearest_neighbor_kernel) - interp1d(x, y, kind='nearest', fill_value="extrapolate")(x_new)) < EPS

#### Линейная интерполяция (1 балл)

Метод линейной интерполяции основан на предположении, что значения между двумя соседними узлами можно аппроксимировать линейной функцией.

Математически ядро линейной интерполяции можно выразить следующим образом:
$$
u(s) =
\begin{cases}
1 - |s|, & |s| \leq 1 \\
0, & |s| > 1
\end{cases}
$$

* В качестве решения вы должны прикрепить функцию ниже. Все пороги должны быть указаны внутри функции.
* Строку (# GRADED CELL: [function_name]) менять **нельзя**. Она будет использоваться при проверке вашего решения.
* Ячейка должна содержать только одну функцию.

In [ ]:
# GRADED CELL: linear_kernel

def linear_kernel(distance):
    """
    Линейное ядро для линейной интерполяции

    :param distance: Массив расстояний до исходных точек (shape: (new_length, original_length))
    :return: Массив весов (shape: (new_length, original_length))
    """
    abs_distance = np.abs(distance)
    ans = np.where(abs_distance<=1.0,abs_distance,1.0)
    return 1 - ans  # ваш код здесь


In [ ]:
a = np.zeros((2,2))
print(a)

In [ ]:
assert np.linalg.norm(interpolate(y, SCALE, linear_kernel) - interp1d(x, y, kind='linear', fill_value="extrapolate")(x_new)) < EPS

#### Кубическая интерполяция (1 балл)

При кубической интерполяции между любыми двумя соседними узлами функция интерполируется кубическим полиномом. Мы рассмотрим один из частных случаев ядра интерполяции, задаваемого полиномом Катмулла-Рома.

Ядро состоит из кусочно-кубических полиномов, определённых на подынтервалах (-2, -1), (-1, 0), (0, 1) и (1, 2). За пределами интервала (-2, 2) интерполяционное ядро равно нулю. В результате этого условия количество сэмплов данных, используемых для вычисления интерполяционной функции в уравнении _(1)_, уменьшается до четырёх.

Математически ядро сплайнов Катмулла-Рома описывается кубическим полиномом для сплайна между узлами:

$$
u(s) =
\begin{cases}
\frac{3}{2} |s|^3 - \frac{5}{2} |s|^2 + 1, & 0 \leq |s| < 1 \\
-\frac{1}{2} |s|^3 + \frac{5}{2} |s|^2 - 4 |s| + 2, & 1 \leq |s| < 2 \\
0, & 2 \leq |s|
\end{cases}
$$



_Note_: Это один из вариантов реализации ядра кубической интерполяции. Подробнее про выбор значений коэффициентов можно найти [здесь](https://www.researchgate.net/publication/3177062_Cubic_convolution_interpolation_for_digital_image_processing_IEEE_Trans_Acoust_Speech_Signal_Process).

* В качестве решения вы должны прикрепить функцию ниже. Все пороги должны быть указаны внутри функции.
* Строку (# GRADED CELL: [function_name]) менять **нельзя**. Она будет использоваться при проверке вашего решения.
* Ячейка должна содержать только одну функцию.

In [ ]:
# GRADED CELL: cubic_kernel

def cubic_kernel(distance):
    """
    Кубическое ядро полинома Катмулла-Рома для кубической интерполяции

    :param distance: Массив расстояний до исходных точек (shape: (new_length, original_length))
    :return: Массив весов (shape: (new_length, original_length))
    """
    abs_distance = np.abs(distance)
    mask1 = np.where(abs_distance<1.0,1,0)
    mask23 = np.where(abs_distance<2.0,1,0)
    mask23 -= mask1
    ans = np.zeros(abs_distance.shape)
    ans += (3/2 * abs_distance**3 - 5/2 * abs_distance**2 + 1)*mask1
    ans += (-1/2 * abs_distance**3 + 5/2 * abs_distance**2 - 4*abs_distance + 2)*mask23
    return ans # ваш код здесь


In [ ]:
assert np.linalg.norm(interpolate(y, SCALE, cubic_kernel) -
                      np.array([2.0, 1.648267009, 1.121418827, 0.5925925925, 0.0877914952, 0.0877914952, 0.5925925925, 1.121418827, 1.648267009, 2.0])
        ) < EPS

#### Визуализация результатов

Для визуализации предлагается использовать элементы управления `plotly`. В ячейке ниже значение `pio.renderers.default` установлено для корректной работы в VSCode.

Вы можете поменять его в зависимости от используемой среды разработки. Подробнее [здесь](https://plotly.com/python/renderers/#:~:text=be%20discussed%20here.-,Setting%20The%20Default%20Renderer,-The%20current%20and).

In [ ]:
import plotly.io as pio
pio.renderers.default = 'colab'

Проверьте, что `plotly` работает корректно. Воспользуйтесь переключателями отображаемых графиков, нажимая в легенде на соответствующие линии:

In [ ]:
import plotly.graph_objects as go

original_length = 8
x_original = np.linspace(0, 2 * np.pi, original_length)
signal = np.sin(x_original)
x_new = np.linspace(0, 2 * np.pi, int(original_length * SCALE))

interpolated_signal_nn = interpolate(signal, SCALE, nearest_neighbor_kernel)
interpolated_signal_linear = interpolate(signal, SCALE, linear_kernel)
interpolated_signal_cubic = interpolate(signal, SCALE, cubic_kernel)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_original, y=signal, mode='markers', name='Исходные точки'))
fig.add_trace(go.Scatter(x=x_new, y=interpolated_signal_nn, mode='lines+markers', name='Ближайший сосед'))
fig.add_trace(go.Scatter(x=x_new, y=interpolated_signal_linear, mode='lines+markers', name='Линейная интерполяция'))
fig.add_trace(go.Scatter(x=x_new, y=interpolated_signal_cubic, mode='lines+markers', name='Кубическая интерполяция'))

fig.update_layout(
    title='Методы интерполяции одномерного сигнала (Plotly)',
    xaxis_title='x',
    yaxis_title='Амплитуда',
    template='plotly_white',
    showlegend=True
)
fig.show()


### Интерполяция изображений

В этой части будут рассмотрены классические методы интерполяции изображений, реализованные в библиотеке OpenCV.

#### Подготовка изображений

Для выполнения этого задания предлагается использовать заранее подготовленные функции в модуле useful_utils.

Для начала давайте попробуем считать изображение и отобразить его в интерактивном окне. Убедитесь, что `plotly` работает корректно, попробуйте с его помощью детально рассмотреть тестовое изображение.

In [ ]:
from useful_utils import read_image, show_images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

gt_path = Path('board.png')
gt = read_image(gt_path)
show_images([gt], ['GT'], figsize=7)



Создадим изображение низкого разрешения, в англоязычной нотации _LQ (Low Quality)_. Самый простой и распространенный способ сделать это — бикубическая интерполяция.

Почитать про более продвинутые пайплайны деградации изображений можно [тут](https://arxiv.org/pdf/2107.10833). В большинстве современных работ используют именно этот метод.

In [ ]:
SCALE = 4

height, width = gt.shape[:2]
new_height, new_width = height // SCALE, width // SCALE
lq = cv2.resize(gt, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

if not os.path.exists("./LQs"):
    os.makedirs("./LQs")
cv2.imwrite("LQs/board.png", lq[..., ::-1])

Далее рассмотрим два метода интерполяции изображений, который могут быть полезны вам в будущем:
1. Метод *Nearest Neighbor (Nearest)* используется для отображения LQ в оригинальном разрешении для сравнения с другими методами;
2. *Bicubic Interpolation(BI)* часто используется в качестве простого бейзлайна для сравнения.

Получите увеличенные этими способами до оригинального размера изображения стенда. Можно использовать библиотеку OpenCV. Рассмотрите детали изображений, используя интерактивный зум, реализованный в функции `show_images`.

In [ ]:
nearest = cv2.resize(lq, (width, height), interpolation=cv2.INTER_NEAREST) # ваш код здесь
bi = cv2.resize(lq, (width, height), interpolation=cv2.INTER_CUBIC) # ваш код здесь

show_images([gt, nearest, bi], ['GT', 'Nearest Neighbor (LQ)', 'Bicubic'], figsize=5)

#### Объективная оценка качества

В качестве основных объективных методов оценки качества изображений в задаче Super-Resolution в статьях обычно используют [PSNR](https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio) и [SSIM](https://ieeexplore.ieee.org/document/1284395).

После выхода статьи [LPIPS](https://openaccess.thecvf.com/content_cvpr_2018/html/Zhang_The_Unreasonable_Effectiveness_CVPR_2018_paper.html), в которой было предложено считать расстояние между изображениями в пространстве признаков, этот метод также стал распространен при оценке качества работы моделей SR.

Более детальный обзор существующих методов оценки качества изображений и видео в задаче Super-Resolution можно найти на странице нашего [бенчмарка](https://videoprocessing.ai/benchmarks/super-resolution-metrics.html).

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

Воспользуйтесь метриками PSNR и SSIM из [torchmetrics](https://docs.pytorch.org/torcheval/main/torcheval.metrics.html) и методом [LPIPS](https://github.com/richzhang/PerceptualSimilarity) для оценки качества изображений, полученных в предыдущем блоке.

Для PSNR используйте аргументы $ reduction$=None и $dim$=[1, 2, 3].

Для SSIM используйте аргументы $ reduction$=None, $return\_full\_image$=False и $return\_contrast\_sensitivity$=False.

Не забудьте, что все методы поддерживают вычисление на GPU ;)

In [ ]:
!pip install lpips
!pip install torchmetrics

In [ ]:
from lpips import LPIPS
from useful_utils import preprocess_image

lpips = LPIPS(net='alex').to(device) # ваш код здесь

from torchmetrics import PeakSignalNoiseRatio as PSNR
psnr = PSNR(reduction = None, data_range=(0,255),dim = [1,2,3]).to(device) # ваш код здесь

from torchmetrics import StructuralSimilarityIndexMeasure as SSIM
ssim =SSIM(reduction=None, return_full_image=False, return_contrast_sensitivity=False).to(device) # ваш код здесь



In [ ]:
from useful_utils import preprocess_image

def metric(im1, im2, metric_fn):
    return metric_fn(preprocess_image(im1, device)[None, ...], preprocess_image(im2, device)[None, ...]).item()

Для визуализации результатов можно воспользоваться еще одной подготовленной функцией:

In [ ]:
from useful_utils import interactive_plot_metrics
img1n = torch.from_numpy(nearest.transpose(2, 0, 1)).unsqueeze(0)
img1b = torch.from_numpy(bi.transpose(2, 0, 1)).unsqueeze(0)
data = {
    'Метод': ['Nearest Neighbor', 'Bicubic'],
    'SSIM↑': [metric(gt, nearest, ssim), metric(gt, bi, ssim)], # ваш код здесь
    'PSNR↑': [metric(gt, nearest, psnr), metric(gt, bi, psnr)], # ваш код здесь
    'LPIPS↓': [metric(gt, nearest, lpips), metric(gt, bi, lpips)] # ваш код здесь
}

interactive_plot_metrics(data)

## Существующие нейросетевые методы SR

В этом блоке вам предстоит запустить одну из современных моделей Image Super-Resolution. Допускается как использование образцов кода для запуска в файле `inference.py` репозитория, так и воспользоваться возможностями Jupyter с помощью синтаксиса [`!command`](http://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-system). Если этого не указано отдельно, используйте параметры моделей по умолчанию.

Для оценки этой части задания необходимо загрузить в систему выход модели на рассмотренном ранее изображении `LQs/board.png`, на котором вы можете проверить себя с помощью assert-ов, и на изображении `LQs/zaya.png`, целевые значения для которого скрыты.

In [ ]:
EPS = 10

def l2_norm(image1, image2):
    # Step 1: Normalize both images to the range [0, 1]
    if image1.max() > 1:
        image1 = image1 / 255.0
    if image2.max() > 1:
        image2 = image2 / 255.0

    # Step 2: Compute L2 norm between the two images
    return np.linalg.norm(image1 - image2)

### SwinIR (4 балла)

Базовая в задачах Image Restoration модель на архитектуре трансформер, с которой часто сравниваются в других работах. Породила целое семейство моделей (HAT, CAT, DAT, etc.)

Пример запуска можно найти в приложенном ноутбуке Colab в репозитории [SwinIR](https://github.com/JingyunLiang/SwinIR).

In [ ]:
!git clone https://github.com/JingyunLiang/SwinIR.git
# ...
# # Download the pre-trained models
!wget https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth -P experiments/pretrained_models
!wget https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth -P experiments/pretrained_models

In [ ]:
!pip install basicsr
!pip install facexlib
!pip install gfpgan
!pip install -r requirements.txt

!python SwinIR/main_test_swinir.py --task real_sr --model_path experiments/pretrained_models/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth --folder_lq LQs --scale 4

In [ ]:
board = read_image("/content/results/swinir_real_sr_x4/board_SwinIR.png")
zaya = read_image("/content/results/swinir_real_sr_x4/zaya_SwinIR.png")
swin_board = board # ваш код здесь
print(swin_board.shape)

with open('asserts/sr/board/swinir.bin', 'rb') as f:
    target = np.frombuffer(f.read(), dtype=np.uint8).reshape(swin_board.shape)

assert l2_norm(swin_board, target) < EPS


if not os.path.exists("./pictures"):
    os.makedirs("./pictures")
cv2.imwrite("pictures/swin_board.png", swin_board)

### Визуализация работы методов

#### Субъективная оценка качества

Визуально сравните качество работы методов, воспользовавшись функцией `show_images`из предыдущего раздела. Включите в сетку $2х2$ $ GT, LQ, BI $ и полученное в результате работы SR изображение.

In [ ]:
show_images([gt, nearest, bi, swin_board], ['GT', 'Nearest Neighbor (LQ)', 'Bicubic', 'SR'], grid=[2,2], figsize=5)

#### Объективная оценка качества

Посчитайте SSIM, PSNR и LPIPS для полученных изображений и визуализируйте результат, воспользовавшись функцией `interactive_plot_metrics` из предыдущего раздела.

In [ ]:
data = {
    'Метод': ['Bicubic', 'Nearest', 'SwinIR'],
    'SSIM↑': [metric(gt, nearest, ssim), metric(gt, bi, ssim), metric(gt, swin_board, ssim)], # ваш код здесь
    'PSNR↑': [metric(gt, nearest, psnr), metric(gt, bi, psnr), metric(gt, swin_board, psnr)], # ваш код здесь
    'LPIPS↓': [metric(gt, nearest, lpips), metric(gt, bi, lpips), metric(gt, swin_board, lpips)] # ваш код здесь
}

interactive_plot_metrics(data)

#### Ваши рассуждения (макс. 2 доп. балла)

Что вы можете сказать о работе нейросетевых методов Image Super-Resolution? Как вы думаете, почему они показывают такое качество?

**Ответ**:

## Нейросетевые артефакты (44 балла)

### Что еще за артефакты? (0 баллов)

Как вы могли заметить, подходы на основе нейронных сетей (в отличие от интерполяций) могут неожиданно порождать визуальные артефакты на некоторых изображениях. Поэтому в данном разделе мы предлагаем вам разработать методы для отдельного обнаружения таких артефактов.

Для ознакомления предлагаем воспользоваться функцией `patch_batch`, считающей значение метрики, например LPIPS, по патчам изображений с некоторым шагом.

Функция `plot_interactive_heatmap` позволит увидеть области, на которых присутствуют наибольшие "по мнению LPIPS" искажения. Обратите внимание, что с помощью ползунка можно менять прозрачность тепловой карты LPIPS, а кнопка в углу переключает изображения на фоне.

In [ ]:
from useful_utils import patch_batch, plot_interactive_heatmap, preprocess_image

In [ ]:
lpips_heatmap = patch_batch(
    gt=preprocess_image(gt, device=device),
    sample=preprocess_image(swin_board, device=device),
    patch_size=32,
    metric=lpips,
    device=device,
    stride=4
).cpu().detach().numpy()
torch.cuda.empty_cache()

In [ ]:
plot_interactive_heatmap(gt, swin_board, lpips_heatmap, "LPIPS heatmap", "SwinIR")

### Детекция областей с артефактами (44 балла)

### Задача:
В данной части задания данные будут иметь следующий формат: датасет состоит из четверок $ (img, mask, gt, has\_artifact) $, где
- $ img $ — тестовое изображение, обработанное SR;
- $ mask $ — бинарная маска предполагаемой области артефакта;
- $ gt $ — оригинальное изображение хорошего качества (без артефактов);
- $ has\_artifact $ — булевская метка, показывающая наличие/отсутствие артефакта (при значении False маска заполнена нулями).

Вам будет предоставлена открытая часть датасета, на которой можно обучить модель предсказания маски артефакта $ mask $ по входным признакам. Оценка качества модели будет осуществляться с помощью метрик IoU и F-Score на закрытой части датасета.

### Данные:

Датасет можно скачать [по ссылке](https://calypso.gml-team.ru:5001/sharing/nzbnCGWiL). Рекомендуем загрузить его на Google Drive, который можно потом примонтировать в colab для удобного доступа к данным.

Разметка датасета находится в файле _labels.csv_. Имена изображений $ img, mask $ и $ gt $ находятся в столбцах _sr_fn_, _mask_fn_ и _gt_fn_ соответственно, метка _has_artifact_ указана в одноименном столбце. Для удобства ниже представлен интерфейс для работы с датасетом (файл artifact_dataset.py).

In [ ]:
# Необходимые библиотеки
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from useful_utils import read_image, show_images
import cv2
from eval_metric import iou
from artifact_dataset import create_dataloader
from torch.utils.checkpoint import checkpoint
import torch.nn.functional as F
import torch
from torchvision import transforms
torch.mps.empty_cache()

In [ ]:
from pathlib import Path
device = 'mps'
dataset_dir = Path('train')
labels_path = dataset_dir / 'labels.csv'

# Создаем даталоадеры, которые будет подавать данные на вход модели
# Разделяем данные на тестовую и валидационную часть
train_loader, val_loader = create_dataloader(dataset_dir, labels_path, batch_size=2, val_size=0.2, random_state=42)

### Архитектура модели

Вам предоставлена простейшая [сверточная нейросеть](https://ru.wikipedia.org/wiki/%D0%A1%D0%B2%D1%91%D1%80%D1%82%D0%BE%D1%87%D0%BD%D0%B0%D1%8F_%D0%BD%D0%B5%D0%B9%D1%80%D0%BE%D0%BD%D0%BD%D0%B0%D1%8F_%D1%81%D0%B5%D1%82%D1%8C) для решения нашей задачи. Вас необходимо улучшить работу данной модели.

#### Правила:

- Нельзя убирать GRADED CELL, менять название класса, менять методы ```save_weight``` и ```load_weigths```, а также убирать поле класса ```threshold``` (но менять его _значение_ можно);
- Нельзя использовать предобученные сети в качестве backbone;
- Нельзя обучаться на сторонних датасетах артефактов SR.

- Можно менять архитектуру и добавлять новые методы.

- На вход подаётся две группы изображений (*img*, *gt*) из батча

In [ ]:
# GRADED CELL: MyModel

def calculate_variance_map(image_batch, kernel_size=7):
    """
    Calculates the local variance map for a batch of images using convolutions.
    
    Args:
        image_batch (torch.Tensor): A batch of images with shape [B, C, H, W].
        kernel_size (int): The size of the sliding window for local variance.

    Returns:
        torch.Tensor: A batch of variance maps with shape [B, 1, H, W].
    """
    grayscale_batch = transforms.Grayscale(num_output_channels=1)(image_batch)
    avg_kernel = torch.ones(1, 1, kernel_size, kernel_size, device=image_batch.device) / (kernel_size * kernel_size)
    local_mean = F.conv2d(grayscale_batch, avg_kernel, padding='same')
    local_mean_sq = F.conv2d(grayscale_batch.pow(2), avg_kernel, padding='same')
    variance_map = local_mean_sq - local_mean.pow(2)
    variance_map = torch.clamp(variance_map, min=0)

    return variance_map

def calculate_local_variance_map(image: np.ndarray, patch_size: int = 7) -> np.ndarray:
    """
    Calculates the local variance for each pixel in an image using a sliding window.

    This function is a core component for both DeSRA and LDL methods. It efficiently
    computes variance using integral images (summed-area tables) which is much
    faster than iterating with a sliding window.

    Args:
        image (np.ndarray): The input grayscale image, with pixel values
                            expected to be in the range [0, 255].
        patch_size (int): The side length of the square patch (window) to
                          calculate variance over. Must be an odd number.

    Returns:
        np.ndarray: A map of the same dimensions as the input image, where each
                    pixel value is the variance of the patch centered on it.
    """
    if patch_size % 2 == 0:
        raise ValueError("Patch size must be an odd number.")

    img_fp = image.astype(np.float64)

    pad_size = patch_size // 2
    img_padded = cv2.copyMakeBorder(img_fp, pad_size, pad_size, pad_size, pad_size, cv2.BORDER_REFLECT)
    
    mean_map = cv2.boxFilter(img_padded, -1, (patch_size, patch_size), normalize=True)
    sq_mean_map = cv2.boxFilter(img_padded**2, -1, (patch_size, patch_size), normalize=True)

    variance_map = sq_mean_map - mean_map**2

    variance_map = variance_map[pad_size:-pad_size, pad_size:-pad_size]
    
    variance_map[variance_map < 0] = 0

    return variance_map


def discriminate_artifacts_ldl(
    gan_sr_img: np.ndarray,
    gt_img: np.ndarray,
    patch_size: int = 7
) -> np.ndarray:

    if gan_sr_img.shape != gt_img.shape:
        raise ValueError("GAN SR and Ground Truth images must have the same dimensions.")

    if len(gan_sr_img.shape) > 2:
        gan_sr_img = cv2.cvtColor(gan_sr_img, cv2.COLOR_BGR2GRAY)
    if len(gt_img.shape) > 2:
        gt_img = cv2.cvtColor(gt_img, cv2.COLOR_BGR2GRAY)

    residual_img = np.abs(gan_sr_img.astype(np.float64) - gt_img.astype(np.float64))

    artifact_map = calculate_local_variance_map(residual_img.astype(np.uint8), patch_size)

    normalized_map = cv2.normalize(artifact_map, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    return normalized_map
# Не меняйте название класса и методов, также класс должен содержать поле threshold

def discriminate_artifacts_ldl_torch(
    gan_sr_tensor: torch.Tensor,
    gt_tensor: torch.Tensor,
    patch_size: int = 7
) -> torch.Tensor:

    if gan_sr_tensor.shape != gt_tensor.shape:
        raise ValueError("GAN SR and Ground Truth tensors must have the same shape.")

    device = gan_sr_tensor.device
    batch_size = gan_sr_tensor.shape[0]
    
    artifact_maps = []

    for i in range(batch_size):
        sr_img_tensor = gan_sr_tensor[i]
        gt_img_tensor = gt_tensor[i]

        sr_img_np = (sr_img_tensor.cpu().detach().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        gt_img_np = (gt_img_tensor.cpu().detach().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        
        if sr_img_np.shape[2] == 1:
            sr_img_np = np.squeeze(sr_img_np, axis=2)
        if gt_img_np.shape[2] == 1:
            gt_img_np = np.squeeze(gt_img_np, axis=2)

        artifact_map_np = discriminate_artifacts_ldl(sr_img_np, gt_img_np, patch_size)

        artifact_map_tensor = torch.from_numpy(artifact_map_np).float().unsqueeze(0) / 255.0
        artifact_maps.append(artifact_map_tensor)

    final_maps = torch.stack(artifact_maps, dim=0).to(device)

    return final_maps

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class MyModel(nn.Module):
    def __init__(self, in_channels=10, threshold=0.5):
        super(MyModel, self).__init__()
        #Threshold
        self.threshold = threshold
        # --- ENCODER ---
        self.Maxpool = nn.MaxPool2d(2)
        self.Conv1 = ConvBlock(in_channels, 24)
        self.Conv2 = ConvBlock(24, 48)
        self.Conv3 = ConvBlock(48, 96)
        self.Conv4 = ConvBlock(96, 192) 
        self.Dropout = nn.Dropout(p=0.3) 

        # --- DECODER ---
        self.Up3 = nn.ConvTranspose2d(192, 96, kernel_size=2, stride=2)
        self.UpConv3 = ConvBlock(192, 96) # Input is 96 + 96

        self.Up2 = nn.ConvTranspose2d(96, 48, kernel_size=2, stride=2)
        self.UpConv2 = ConvBlock(96, 48)  # Input is 48 + 48

        self.Up1 = nn.ConvTranspose2d(48, 24, kernel_size=2, stride=2)
        self.UpConv1 = ConvBlock(48, 24)   # Input is 24 + 24

        self.Conv_1x1 = nn.Conv2d(24, 1, kernel_size=1)

    def forward(self, x):
        img = x['img']
        gt = x['gt']
        diff = torch.abs(img-gt)
        ldl = discriminate_artifacts_ldl_torch(img,gt)
        model_input = torch.cat([img, gt, diff,ldl], dim=1)

        # --- FORWARD PASS WITH SKIP CONNECTIONS ---
        x1 = self.Conv1(model_input)
        x2 = self.Maxpool(x1); x2 = self.Conv2(x2)
        x3 = self.Maxpool(x2); x3 = self.Conv3(x3)
        x4 = self.Maxpool(x3); x4 = self.Conv4(x4)

        d3 = self.Up3(x4); d3 = torch.cat((x3, d3), dim=1); d3 = self.UpConv3(d3)
        d2 = self.Up2(d3); d2 = torch.cat((x2, d2), dim=1); d2 = self.UpConv2(d2)
        d1 = self.Up1(d2); d1 = torch.cat((x1, d1), dim=1); d1 = self.UpConv1(d1)

        return self.Conv_1x1(d1)

    def save_weights(self, path):
        torch.save(self.state_dict(), path)

    def load_weights(self, path, device=None):
        self.load_state_dict(torch.load(path, map_location=device))

### Цикл обучения

Ниже представлен рабочий шаблон для цикла обучения. Его тоже улучшить, подобрав функцию потерь, optimizer, learning rate и обновив стратегию обучения в целом.

In [ ]:
#cool loss
class DiceLoss(nn.Module):
    """
    Dice Loss, directly optimizes the Dice Coefficient (closely related to IoU).
    Helps with class imbalance.
    """
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)

        return 1 - dice

class FocalLoss(nn.Module):
    """
    Focal Loss, focuses training on hard-to-classify examples.
    An enhancement of BCEWithLogitsLoss.
    """
    def __init__(self, alpha=0.8, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

class JaccardLoss(nn.Module):
    def __init__(self, smooth=1e-6):

        super(JaccardLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        total = (probs + targets).sum()
        union = total - intersection

        jaccard = (intersection + self.smooth) / (union + self.smooth)

        return 1 - jaccard



def lovasz_grad(gt_sorted):
    """
    Computes gradient of the Lovász-Hinge loss.
    """
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:  # cover 1-pixel case
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    """
    Binary Lovász hinge loss
      logits: [P] Variable, logits at each pixel (between -\infty and +\infty)
      labels: [P] Tensor, binary ground truth labels (0 or 1)
    """
    if len(labels) == 0:
        # only void pixels, the gradients should be 0
        return logits.sum() * 0.
    
    signs = 2. * labels.float() - 1.
    errors = (1. - logits * signs)
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm = perm.data
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), grad)
    return loss


class LovaszHingeLoss(nn.Module):
    """
    Lovász-Hinge Loss, a direct optimization of the Jaccard index (IoU) for
    binary segmentation. It is a convex surrogate for the IoU metric, making
    the optimization landscape smoother and often leading to better performance,
    especially on datasets with complex object structures.
    """
    def __init__(self, per_image=True):
        """
        Initializes the LovaszHingeLoss module.
        Args:
            per_image (bool): If True, computes the loss for each image in the batch
                              and averages them. If False, flattens the whole batch
                              and computes the loss once.
        """
        super(LovaszHingeLoss, self).__init__()
        self.per_image = per_image

    def forward(self, logits, targets):
        """
        Calculates the Lovász-Hinge Loss.
        Args:
            logits (torch.Tensor): Raw model output (N, 1, H, W) or (N, H, W).
            targets (torch.Tensor): Ground truth (N, 1, H, W) or (N, H, W).
        Returns:
            torch.Tensor: The calculated Lovász-Hinge loss.
        """
        if self.per_image:
            loss = torch.stack([
                lovasz_hinge_flat(
                    logit.view(-1),
                    target.view(-1)
                )
                for logit, target in zip(logits, targets)
            ])
            return loss.mean()
        else:
            return lovasz_hinge_flat(
                logits.view(-1),
                targets.view(-1)
            )


In [ ]:
from tqdm import tqdm

def train_model(model, num_epochs, dataloader, val_loader, device, save_path):
    """
    Advanced training loop with a combined loss, AdamW optimizer,
    and a learning rate scheduler.
    """
    #super cool loss
    focal_loss = FocalLoss().to(device)
    dice_loss = DiceLoss().to(device)
    lhl = LovaszHingeLoss().to(device)
    jl = JaccardLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    best_val_iou = 0
    
    history = {'train_loss': [], 'train_iou': [], 'val_loss': [], 'val_iou': []}
    print(f'Training begins | {num_epochs} epochs | Device: {device}')
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        total_train_iou = 0.0
        num_batches = 0

        pbar = tqdm(dataloader)
        for batch in pbar:
            for key in batch.keys():
                batch[key] = batch[key].to(device)
            img, mask = batch['img'], batch['mask']

            model_input = {'img': img, 'gt' : batch['gt']}
            pred_logits = model(model_input)

            # Calculate the combined loss
            
            loss_focal = focal_loss(pred_logits, mask)
            loss_dice = dice_loss(pred_logits, mask)
            lhl_loss = lhl(pred_logits,mask)
            jl_loss = jl(pred_logits,mask)
            loss = jl_loss + loss_focal
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            with torch.no_grad(): # Disable gradients for metric calculation
                bin_mask = (torch.sigmoid(pred_logits) >= model.threshold).float()
                total_train_iou += iou(bin_mask, batch['mask'])
            num_batches += 1
            pbar.set_description(f'Epoch {epoch+1}/{num_epochs} | Avg Loss: {(total_train_loss / num_batches):.4f}')
        model.eval()
        total_val_loss = 0.0
        total_val_iou = 0.0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Validation]'):
                for key in batch.keys():
                    batch[key] = batch[key].to(device)
                img, mask, gt = batch['img'], batch['mask'], batch['gt']

                model_input = {'img': img, 'gt': gt}
                pred_logits = model(model_input)
                
                # Calculate validation loss
                val_loss_focal = focal_loss(pred_logits, mask)
                val_loss_dice = dice_loss(pred_logits, mask)
                val_loss_lhl = lhl(pred_logits,mask)
                val_loss_jl = jl(pred_logits,mask)
                val_loss = val_loss_jl + val_loss_focal
                total_val_loss += val_loss.item()
                
                # Calculate validation IoU
                bin_mask = (torch.sigmoid(pred_logits) >= model.threshold).float()
                total_val_iou += iou(bin_mask, mask).item()

        # --- EPOCH END LOGGING & SAVING ---
        avg_train_loss = total_train_loss / len(train_loader)
        avg_train_iou = total_train_iou / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        avg_val_iou = total_val_iou / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['train_iou'].append(avg_train_iou)
        history['val_loss'].append(avg_val_loss)
        history['val_iou'].append(avg_val_iou)
        
        scheduler.step()

        print(f"\nEpoch {epoch+1} Summary: Train Loss: {avg_train_loss:.4f}, Train IoU: {avg_train_iou:.4f} | "
              f"Val Loss: {avg_val_loss:.4f}, Val IoU: {avg_val_iou:.4f}")

        # Save the model only if validation IoU has improved
        if avg_val_iou > best_val_iou:
            best_val_iou = avg_val_iou
            print(f"🚀 New best validation IoU: {best_val_iou:.4f}. Saving model to {save_path}")
            model.save_weights(save_path)

    print(f'\nTraining ends. Best validation IoU achieved: {best_val_iou:.4f}')
    return history

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
save_path = './base_model.pth'
super_val_iou = 0
# for i in  [0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95]:
    # save_path = f"./base_model{i}.pth"
model = MyModel().to(device)
history = train_model(model, num_epochs=50, dataloader=train_loader, val_loader = val_loader, device=device,save_path=save_path)


# Не забудьте сохранить веса! Их нужно будет отправить в тестовую систему.
# model.save_weights(save_path)

print(f'Weights saved to {save_path}')

In [ ]:
# Проверьте, что сохраненные веса корректно загружаются в модель
weights_path = './base_model.pth'
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
load_model = MyModel().to(device)
load_model.load_weights(weights_path, device)

### Визуализация

Посмотрим, чему обучилась ваша модель. Функции в следующей ячейке получают предсказание вашей модели на тестовом изображении и визуализируют маску артефакта.

**_На заметку:_**
> Можно менять тестовое изображение, передав путь до него в ```visualize_probability_mask``` в аргумент ```image_path```.

> Если модель выделила много лишних деталей, попробуйте повысить ```model.threshold```.

In [ ]:
# Вспомогательные функции
def mask2img(img):
    img = img.permute(1, 2, 0).detach().cpu().numpy().clip(0, 1)
    img = (img * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    return img

def get_test_batch(test_img_path, device):
    gt_path = test_img_path[:-4] + '@gt.png'

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.ConvertImageDtype(torch.float32)
    ])

    test_img =  Image.open(test_img_path).convert('RGB')
    test_img = transform(test_img).to(device)

    test_gt =  Image.open(gt_path).convert('RGB')
    test_gt = transform(test_gt).to(device)

    return {'img': test_img,
            'gt': test_gt}

def get_mask(mask_path):
    mask_img =  Image.open(mask_path).convert('L')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.ConvertImageDtype(torch.float32)
    ])
    mask_img = transform(mask_img).to(device)

    return mask_img

In [ ]:
def visualize_probability_mask(model, image_path=None):
    device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
    if image_path is None:
        test_img_path = './test_image.png'
    else:
        test_img_path = image_path

    test_batch = get_test_batch(test_img_path, device)
    test_batch['img'] = test_batch['img'].unsqueeze(0)
    test_batch['gt'] = test_batch['gt'].unsqueeze(0)
    pred_mask = model(test_batch)
    pred_mask = pred_mask.squeeze(0)
    show_images([read_image(test_img_path), mask2img(pred_mask)],
            ['Тестовое изображение', f'Маска артефакта'])

In [ ]:
visualize_probability_mask(load_model)

### Подбор порога
Как можно заметить, модель предсказывает _вероятность артефакта_ в каждом пикселе. Нам нужно, чтобы она возвращала бинарную маску, в которой единица означает наличие артефакта, ноль - его отсутствие. Для этого можно воспользоваться **отсечением по порогу (threshold)** :

$$
\text{binary\_ mask}[x, y] =
\begin{cases}
1 & \text{predicted\_ mask}[x, y] > \text{threshold} \\
0 & \text{predicted\_ mask}[x, y] <= \text{threshold}
\end{cases}
$$


Поле ```model.threshold``` вашей модели отвечает за порог. Попробуйте визуализировать бинарную маску с помощью функции в следующей ячейке и подобрать параметр ```threshold```.

# Важно!
> После подбора оптимального порога не забудьте перенести лучшее значение ```threshold``` в ячейку с GRADED CELL с определением класса ```MyModel```. **Иначе ваш порог не попадет в тестовую систему!**

In [ ]:
def visualize_binary_mask(model, image_path=None):
    device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
    if image_path is None:
        test_img_path = './test_image.png'
    else:
        test_img_path = image_path
    mask_path = test_img_path[:-4] + '@mask.png'

    test_batch = get_test_batch(test_img_path, device)
    test_batch['img'] = test_batch['img'].unsqueeze(0)
    test_batch['gt'] = test_batch['gt'].unsqueeze(0)
    pred_mask = model(test_batch)
    pred_mask = pred_mask.squeeze(0)
    pred_mask = (pred_mask >= model.threshold).float()

    iou_val = iou(pred_mask, get_mask(mask_path))
    show_images([read_image(test_img_path), mask2img(pred_mask), read_image(mask_path)],
            ['Тестовое изображение', f'Маска артефакта | IoU {iou_val:.3f}', 'GT маска'])

In [ ]:
visualize_binary_mask(load_model)

### Валидация метрики

В тестирующей системе ваша модель будет проверяться на закрытой части датасета при помощи метрик *[intersection over union (IoU)](https://en.wikipedia.org/wiki/Jaccard_index)* и [*F-Score*](https://en.wikipedia.org/wiki/F-score). Вы можете проверить работу вашей модели на валидационном датасете с помощью кода в следующей ячейке.

Обратите внимание, что ```threshold``` является обязательным полем класса модели.

In [ ]:
def clean_mask(batch_of_masks, kernel_size=3):
    """
    Applies morphological opening and closing to a batch of binary masks.
    """
    masks_np = batch_of_masks.cpu().numpy().astype(np.uint8)    
    cleaned_masks = []
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    for i in range(masks_np.shape[0]):
        mask = masks_np[i, 0, :, :] 
        opened_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
        closed_mask = cv2.morphologyEx(opened_mask, cv2.MORPH_CLOSE, kernel, iterations=1)
        cleaned_masks.append(closed_mask)
        
    cleaned_batch = torch.from_numpy(np.array(cleaned_masks)).float().unsqueeze(1)    
    return cleaned_batch.to(batch_of_masks.device)



In [ ]:
def post_process_mask_tensor(batch_bin_mask):
        
    batch_bin_mask_cpu = batch_bin_mask.detach().cpu()
    output_device = batch_bin_mask.device 
    
    batch_size = batch_bin_mask_cpu.shape[0]
    channel_size = batch_bin_mask_cpu.shape[1] 
    
    processed_masks_list = []

    for i in range(batch_size):
        bin_mask_2d_tensor = batch_bin_mask_cpu[i].squeeze() 
    
        h, w = bin_mask_2d_tensor.shape
        
        binary_mask_np = bin_mask_2d_tensor.numpy().astype(np.uint8)

        contours, _ = cv2.findContours(binary_mask_np,
                                       cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)

        processed_mask_np = np.zeros_like(binary_mask_np) 
        for c in contours:
            x, y, w_rect, h_rect = cv2.boundingRect(c)
            cv2.rectangle(processed_mask_np,
                          (x, y),           
                          (x + w_rect, y + h_rect), 
                          (1),              
                          -1)              
        processed_mask_tensor = torch.from_numpy(processed_mask_np).view(channel_size, h, w)
        processed_masks_list.append(processed_mask_tensor)

    try:
        final_batch_tensor = torch.stack(processed_masks_list).to(output_device).float()
    except RuntimeError as e:
        print(f"ERROR: Could not stack tensors. This means images *within the same batch* have different sizes.")
        print(f"Looping over batch size {batch_size}, with shapes: {[m.shape for m in processed_masks_list]}")
        raise e
    
    return final_batch_tensor

In [ ]:
# Пример валидации на тестовой части
from torchmetrics.classification import BinaryF1Score
model = load_model

model.eval()
total_iou = 0
num_batches = 0

f1_metric = BinaryF1Score(threshold=model.threshold).to(device)
f1_metric.reset()
# for i in  [0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95]:
#     weights_path = f'./base_model{i}.pth'
#     device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
#     load_model = MyModel(i).to(device)
#     load_model.load_weights(weights_path, device)
with torch.no_grad():
    for batch in val_loader:
        for key in batch.keys():
            batch[key] = batch[key].to(device)
        img, mask, gt, has_art = batch['img'], batch['mask'], batch['gt'], batch['has_artifact'].float()
        model_input = {'img': img, 'gt': gt}
        load_model = MyModel().to(device)
        load_model.load_weights(weights_path, device)
        load_model.eval() # Set model to evaluation mode
        pred_mask = load_model(model_input)
        bin_mask = (torch.sigmoid(pred_mask) >= model.threshold).float() 
        cleaned_bin_mask = clean_mask(bin_mask) 
        pred_has_art = (cleaned_bin_mask.sum(dim=(1, 2, 3)) > 0).float()
        f1_metric.update(pred_has_art, has_art)
        total_iou += iou(cleaned_bin_mask, mask)
        num_batches += 1
f1_score = f1_metric.compute().item()
avg_iou = (total_iou / num_batches).item()
print(f'\nAverage IoU: {avg_iou:.4f} Total F-Score: {f1_score:.4f} final score: {avg_iou*0.75+0.25*f1_score}')

## Выводы

Поэкспериментируйте с обученной вами метрикой, позапускайте её на разных изображениях. Напишите свои выводы из экспериментов и возможные идеи для улучшения.

**Ответ:**

Лучший лосс, это обратный IoU. Идей уже особо нет, на ничего стоящего видеопамяти не хватает, а пост обработка не сработала.